In [1]:
import pandas as pd
from pathlib import Path

raw_path = Path("../datos/raw")

archivos = list(raw_path.glob("*.csv"))

len(archivos), archivos

(5,
 [WindowsPath('../datos/raw/2025-1239042025-121809promedio-contribuciones-cientificas-por-tesis_2025-05-27_curso_academico.csv'),
  WindowsPath('../datos/raw/2025-1239052025-121811promedio-contribuciones-cientificas-por-tesis_2025-05-27_curso_academico.csv'),
  WindowsPath('../datos/raw/2025-1239052025-121812promedio-contribuciones-cientificas-por-tesis_2025-05-27_curso_academico.csv'),
  WindowsPath('../datos/raw/2025-1239062025-121813promedio-contribuciones-cientificas-por-tesis_2025-05-27_curso_academico.csv'),
  WindowsPath('../datos/raw/2026-1209042026-090306promedio-contribuciones-cientificas-por-tesis_2026-05-27_curso_academico.csv')])

In [10]:

dfs = []

for i in archivos:
    # Los archivos de 2025 están en latin-1 y el de 2026 en UTF-8
    try:
        df = pd.read_csv(i, sep=";", encoding="utf-8-sig")
    except UnicodeDecodeError:
        df = pd.read_csv(i, sep=";", encoding="latin-1")
    
    # Normalizar el nombre de CURSO por si tiene BOM
    df.columns = df.columns.str.replace("ï»¿", "", regex=False)
    
    dfs.append(df)

df_final = pd.concat(dfs, ignore_index=True)

df_final

,CURSO,COD_RUCT,TITULACION,IND_PROM_CONTR_CIEN_TESIS_DOC
0,2023-24,5600043,PROGRAMA DE DOCTORADO EN ADMINISTRACIÓN Y DIRE...,"7,43"
1,2023-24,5601427,"PROGRAMA DE DOCTORADO EN ARQUITECTURA, EDIFICA...",5
2,2023-24,5600038,"PROGRAMA DE DOCTORADO EN ARQUITECTURA, EDIFICA...","7,86"
3,2023-24,5600428,PROGRAMA DE DOCTORADO EN ARTE: PRODUCCIÓN E IN...,"13,34"
4,2023-24,5600046,"PROGRAMA DE DOCTORADO EN AUTOMÁTICA, ROBÓTICA ...","5,29"
...,...,...,...,...
151,2024-25,5600042,PROGRAMA DE DOCTORADO EN RECURSOS Y TECNOLOGÍA...,"9,33"
152,2024-25,5600035,PROGRAMA DE DOCTORADO EN SISTEMAS PROPULSIVOS ...,"5,53"
153,2024-25,5600356,PROGRAMA DE DOCTORADO EN TÉCNICAS EXPERIMENTAL...,NaN
154,2024-25,5600596,PROGRAMA DE DOCTORADO EN TECNOLOGÍAS PARA LA S...,"7,95"


In [11]:
df_final["CURSO"].value_counts()

CURSO
2023-24    32
2022-23    32
2024-25    32
2021-22    30
2020-21    30
Name: count, dtype: int64

In [12]:
contribuciones_por_carrera = df_final.pivot(
    index=["COD_RUCT", "TITULACION"],
    columns="CURSO",
    values="IND_PROM_CONTR_CIEN_TESIS_DOC"
).reset_index()

contribuciones_por_carrera

CURSO,COD_RUCT,TITULACION,2020-21,2021-22,2022-23,2023-24,2024-25
0,5600035,PROGRAMA DE DOCTORADO EN SISTEMAS PROPULSIVOS ...,"6,22",6,"5,32",5,"5,53"
1,5600036,PROGRAMA DE DOCTORADO EN CIENCIA Y TECNOLOGÍA ...,"6,13","6,83",10,"9,15","10,17"
2,5600037,PROGRAMA DE DOCTORADO EN INFORMÁTICA,"9,56","8,28","7,63","8,67","6,92"
3,5600038,"PROGRAMA DE DOCTORADO EN ARQUITECTURA, EDIFICA...","11,88","8,5","8,67","7,86","8,37"
4,5600039,PROGRAMA DE DOCTORADO EN ECONOMÍA AGROALIMENTARIA,4,"3,5",5,"8,57","8,17"
5,5600040,PROGRAMA DE DOCTORADO EN INGENIERÍA Y PRODUCCI...,"10,82","8,13","9,96","8,46","6,89"
6,5600041,PROGRAMA DE DOCTORADO EN INGENIERÍA ELECTRÓNICA,5,16,9,"4,5","4,5"
7,5600042,PROGRAMA DE DOCTORADO EN RECURSOS Y TECNOLOGÍA...,"7,86",7,"6,5","7,7","9,33"
8,5600043,PROGRAMA DE DOCTORADO EN ADMINISTRACIÓN Y DIRE...,"6,75","8,44",9,"7,43","6,13"
9,5600045,PROGRAMA DE DOCTORADO EN INDUSTRIAS DE LA COMU...,"9,4","4,55","5,67","6,1","4,46"


In [14]:
contribuciones_por_carrera.to_csv(
    "../datos/processed/contribuciones_por_carrera.csv",
    sep=";",
    encoding="utf-8-sig",
    index=False
)